# StreamVault back-catalog analysis — the narrative

**This notebook contains no analysis.** Every number below is computed by a function
in `pipeline/` and every table is an artefact written by `run_pipeline.py`.
That is deliberate: if a figure existed only in a cell here, it would not survive
Adjoa's rerun next quarter, and nobody could tell which cells had been run in which
order to produce it.

So read this as a guided tour of work that happens elsewhere. To change a number,
change the code or the settings — not a cell.

**Start the pipeline first**, if you have not already:

```
python run_pipeline.py
```

The one-page answer is in [`reports/memo.md`](../reports/memo.md). This notebook is
for someone who wants to see how it was reached.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# This notebook lives in explore/, so the repository root is one level up.
# It imports the pipeline and computes nothing of its own: every number below
# comes from the same code run_pipeline.py runs, so the narrative can never
# disagree with the artefacts (ADR-027).
sys.path.insert(0, str(Path.cwd().parent))

from pipeline.config import load_settings
from pipeline.extract import load_fetch_log, load_ledger, reconciliation_summary

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

settings = load_settings()
PROCESSED = settings.path(settings["paths"]["processed_dir"])
INTERIM = settings.path(settings["paths"]["interim_dir"])
REPORTS = settings.path(settings["paths"]["reports_dir"])
print("settings loaded and validated")

## As of when?

The analysis reads a cache, so "today" is not the answer. The figures describe the
data as it stood when it was fetched, and that date is stamped on every cached
response — it is what the memo quotes.

In [ ]:
fetch_log = load_fetch_log(PROCESSED / "fetch_log.csv")
dates = fetch_log["fetched_at"].astype(str).str.slice(0, 10)
print("data as of:", dates.min(), "to", dates.max() if dates.max() != dates.min() else "(single day)")
print("films fetched:", int((fetch_log["http_status"] == 200).sum()))

## Stage 1 — every title on the list, and what happened to it

The ledger holds **one row per row of the source file, forever**. Nothing is ever
dropped; a row that cannot be used gets a status and a reason. This is what makes the
memo's "127 received, 112 analysed" sentence checkable rather than asserted.

In [ ]:
ledger = load_ledger(PROCESSED / "id_ledger.csv")
print(f"{len(ledger)} rows in the ledger ({ledger.attrs.get('data_rows', len(ledger) - 1)} data rows)")
reconciliation_summary(ledger)

In [ ]:
# The rows that did not make it, with the reason recorded on each.
unusable = ledger.loc[~ledger["status"].isin(["fetched", "header_row"])]
unusable[["source_row", "raw_value", "status", "reason"]]

### The one that is worth dwelling on

Row 95 holds `tt0468569` — an IMDb identifier in a column of TMDb ids. The obvious fix
is to resolve it and add the film back. It resolves to *The Dark Knight*, **which is
already on the list twice**, so recovering it would have put a third copy of one
franchise film into every comparison.

The procedure that handles this is in `fetch.resolve_foreign_ids`, so next quarter's
file gets the same treatment without anyone remembering this paragraph.

## Stages 3 and 4 — four tables, one grain each

A grain mistake never raises. It produces a plausible number: a sum over the cast
table counts a film once per actor. So each table's grain is asserted, not assumed.

In [ ]:
tables = {name: pd.read_csv(INTERIM / f"{name}.csv")
          for name in ("films", "cast", "crew", "genres")}
pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in tables.items()
])

In [ ]:
# The join is proven rather than trusted: validate="one_to_one" and indicator=True
# on every merge, with the counts written out.
pd.read_csv(PROCESSED / "join_report.csv")

## Stage 5 — real extremes are not errors

*Star Wars* returned 70 times its budget. A budget of 5 would be a typo. Both are far
outside anything typical, and no threshold on its own separates them — so every flagged
value carries a **verdict and a reason**, and nothing is deleted.

In [ ]:
flags = pd.read_csv(PROCESSED / "flags.csv")
print("verdicts:", flags["verdict"].value_counts().to_dict())
flags[["title", "flag", "verdict", "why"]].head(10)

**Every one of the eight extremes is pre-2000.** That is why they are flagged and kept
rather than clipped: winsorising would have removed 20% of the pre-2000 films and none
of the modern ones — an "outlier treatment" that only ever fires on one era is a thumb
on the scale of the comparison Finance is already sceptical about.

In [ ]:
# Why pre- and post-2000 money is not the same unit.
pd.read_csv(PROCESSED / "era_comparability.csv")

## Stage 6 — does the answer depend on the judgement calls?

Each cleaning choice is a setting, so the rejected alternative is *runnable*. This
table re-runs the headline comparison under each one. Read `franchise_ahead` first
("is the recommendation safe?") and `gap` second ("how precisely may I state it?").

In [ ]:
pd.read_csv(PROCESSED / "sensitivity.csv")

The direction holds in every run. The **size** of the lead swings 3.4x, driven almost
entirely by where the nine lone collection members go — which is why the memo gives a
range and names the choice it depends on.

## Stage 7 — the comparison, both windows

Counts beside every statistic. That matters more than usual here: the standalone group
is 25 films, and 12 after 2000.

In [ ]:
for window in ("all_years", "modern"):
    print(f"--- {window} ---")
    display(pd.read_csv(PROCESSED / f"kpi_{window}.csv")[[
        "franchise_group_strict", "films", "median_roi", "floor_roi",
        "iqr_roi", "mean_roi", "hit_rate_primary",
    ]])

### The line that looks like a formatting preference

Compare `mean_roi` with `median_roi` above. **On the mean, standalones win. On the
median, franchises win.** The three exceptional pre-2000 films sit disproportionately
in the smaller group, so the mean reverses the recommendation. The memo reports medians
and says so unprompted.

In [ ]:
# Does the answer change between windows? Every measure, not just the median.
pd.read_csv(PROCESSED / "kpi_flip_check.csv")

`iqr_roi` is marked `better_is: lower` — a tighter group is the safer buy. It is the
one measure that flips, and the margin is 0.17 on 12 films, so the defensible statement
is that the franchise spread advantage *disappears* after 2000, not that standalones
became safer.

In [ ]:
# Inside a franchise: are the middle entries the weak ones?
pd.read_csv(PROCESSED / "kpi_position_in_franchise.csv")

Pooled, that looks like a 43% decline from opener to final entry. But openers are
disproportionately older films, and older films look more profitable for reasons that
have nothing to do with quality. Within one era the decline is **14%**, and middle
entries sit *above* last entries — so the team is right about the direction and wrong
about which entries are weak.

In [ ]:
# The shortlist, with the runners-up it beat.
display(pd.read_csv(PROCESSED / "shortlist_franchises.csv")[[
    "collection_name", "films_on_list", "median_roi", "floor_roi", "bundle_roi", "selected",
]])
display(pd.read_csv(PROCESSED / "shortlist_standalones.csv")[[
    "title", "release_year", "roi", "revenue", "franchise_status", "selected",
]])

Two of the five "standalone" picks are **lone collection members** — films that belong
to a series of which only one instalment is on offer. They are legitimate picks, but
presenting *Jaws* as a standalone without that qualification invites a fair objection,
so the label travels with the row.

In [ ]:
# Talent: the film count stays visible on every row, below the minimum as well as above.
directors = pd.read_csv(PROCESSED / "kpi_talent_directors.csv")
directors.loc[directors["meets_min_films"], [
    "person_name", "films", "median_roi", "floor_roi", "hit_rate_roi",
    "above_overall_median",
]]

In [ ]:
# And the sensitivity: does the reliable list survive widening from leads to top-3 billed?
widened = pd.read_csv(PROCESSED / "kpi_talent_sensitivity.csv")
print(f"{len(widened)} reliable names; "
      f"{int(widened['appears_only_when_widened'].sum())} appear only when widened")
widened.head(10)

## Stage 8 — where the time goes, and what it will cost at 500 titles

The boundary, stated before any number: **stages 3 to 7, warm cache, one machine,
single-threaded. Acquisition excluded** — timing it would measure the network, not this
code — and reported beside the budget rather than inside it.

In [ ]:
display(pd.read_csv(PROCESSED / "time_budget.csv"))
display(pd.read_csv(PROCESSED / "projection.csv"))

The projection is **measured, not multiplied**: stages 3–7 are re-run over subsets of
the real payloads at four sizes and a scaling exponent is fitted per stage. The
all-linear figure is published beside the fitted one so the conclusion does not rest on
the fit being right.

## Stage 9 — checking our own work against the live source

A random sample of 12 from 112 would usually miss every film that matters. So the
sample is stratified: the shortlisted titles, **all** the extremes, Yaw's window, and an
unbiased control.

In [ ]:
display(pd.read_csv(PROCESSED / "spotcheck_sample.csv"))
display(pd.read_csv(PROCESSED / "spotcheck_summary.csv"))

What this establishes is **fidelity** — that the values the analysis used are the values
the source holds. It does *not* establish that the source is correct, and it is a weak
test of drift over time, because the cache was fetched hours before the check. The full
write-up, generated by the stage itself, is in
[`reports/spotcheck.md`](../reports/spotcheck.md).

## Stage 10 — the gate, then the memo

Four checks run before the memo is written. If any fails, **no memo and no shortlist**
are produced. Adjoa reruns this quarterly and sometimes live; the failure this protects
against is a confident number from a broken run, in front of a board, with nothing to
signal that it is wrong.

In [ ]:
pd.read_csv(PROCESSED / "gate.csv")

## Where to read next

| | |
|---|---|
| [`reports/memo.md`](../reports/memo.md) | **The answer.** One page, business language. |
| [`docs/findings.md`](../docs/findings.md) | Every problem found, ranked by how much it moves the recommendation. |
| [`docs/adr/`](../docs/adr/) | Every judgement call in five parts, including what it costs if it is wrong. |
| [`reports/spotcheck.md`](../reports/spotcheck.md) | The live re-check. |
| [`reports/improvement.md`](../reports/improvement.md) | The one addition beyond the brief, and why that one. |